# EVASPA Evaporative Fraction generic edge and model

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import pandas as pd
import xarray as xr

In [ ]:
from sweat.evaspa.edge import (
    DoubleLinearEdge,
    FlatEdge,
    FlatLinearEdge,
    FlatPercentileEdge,
    FlatRegressionEdge,
    LinearEdge,
    ParabolicEdge,
    ThresholdLinearEdge,
)
from sweat.evaspa.ef import EFModel

In [ ]:
from sweat.extra.plot_notebook import plot_edges as plot

### Generate test data 

In [ ]:
def setup(
    var_min: float,
    var_max: float,
    dry_c0: float,
    wet_c0: float,
    dry_c1: float,
    wet_c1: float,
    dry_c2: float = 0.0,
    wet_c2: float = 0.0,
    dry_cut: float = 0.0,
    wet_cut: float = 0.0,
    size: int = 80,
    seed: int = 0,
    law: str = "uniform",
) -> tuple[npt.NDArray, npt.NDArray]:
    """
    Generate data for tests
    """
    # Seed
    np.random.seed(seed)

    # Edge definitions
    def dry_edge(v: float) -> float:
        return dry_c2 * v * v + dry_c1 * v + dry_c0

    def wet_edge(v: float) -> float:
        return wet_c2 * v * v + wet_c1 * v + wet_c0

    if law == "nonuniform":
        df = pd.DataFrame(
            data={
                "var": np.random.triangular(
                    left=var_min, mode=var_min, right=var_max, size=size * size
                )
            }
        )
    else:
        df = pd.DataFrame(
            data={
                "var": np.random.uniform(
                    low=var_min, high=var_max, size=size * size
                )
            }
        )
    df["lst"] = df.apply(
        lambda x: np.random.uniform(wet_edge(x["var"]), dry_edge(x["var"])),
        axis=1,
    )
    dry_lst_cut = dry_edge(dry_cut)
    wet_lst_cut = wet_edge(wet_cut)
    df.loc[(df["var"] <= dry_cut) & (df["lst"] > dry_lst_cut), "lst"] = (
        dry_lst_cut
    )
    df.loc[(df["var"] <= wet_cut) & (df["lst"] < wet_lst_cut), "lst"] = (
        wet_lst_cut
    )
    lst_arr = df["lst"].to_numpy().reshape(size, -1)
    var_arr = df["var"].to_numpy().reshape(size, -1)
    return var_arr, lst_arr

## Edge definition

Flat edge

In [ ]:
var, lst = setup(
    var_min=0.0,
    var_max=0.6,
    dry_c0=330.0,
    dry_c1=-13.0,
    wet_c0=300,
    wet_c1=30,
)

In [ ]:
plot(var, lst)

In [ ]:
edge_sup = FlatEdge.model_validate_json('{"position":"top"}')

In [ ]:
print(edge_sup)

In [ ]:
edge_sup.fit(var, lst)

In [ ]:
print(edge_sup)

In [ ]:
edge_inf = FlatEdge.model_validate_json('{"position":"bottom"}')
edge_inf.fit(var, lst)
print(edge_sup)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

Flat edge with percentile

In [ ]:
var, lst = setup(
    var_min=0.0,
    var_max=0.6,
    dry_c0=330.0,
    dry_c1=-13.0,
    wet_c0=300,
    wet_c1=30,
)
plot(var, lst)

In [ ]:
config_sup = '{"position":"top","percentile":0.1,"selection":"median"}'
edge_sup = FlatPercentileEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","percentile":0.5,"selection":"median"}'
edge_inf = FlatPercentileEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

In [ ]:
config_sup = '{"position":"top","nb_points": 10, "selection":"median"}'
edge_sup = FlatPercentileEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","nb_points": 10, "selection":"median"}'
edge_inf = FlatPercentileEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

Linear edge

In [ ]:
var, lst = setup(
    var_min=0.0,
    var_max=0.6,
    dry_c0=330.0,
    dry_c1=-13.0,
    wet_c0=300,
    wet_c1=30,
)
plot(var, lst)

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median"}'
edge_sup = LinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median"}'
edge_inf = LinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"percentile_limit": 100, "selection":"median"}'
edge_sup = LinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"percentile":2,"percentile_limit":100, "selection":"median"}'
edge_inf = LinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"nb_points":10,"selection":"median"}'
edge_sup = LinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"nb_points":10,"selection":"median"}'
edge_inf = LinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

In [ ]:
config_sup = '{"position":"top","interval_type":"size","interval_size":0.1,"nb_points":10,"selection":"median"}'
edge_sup = LinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"size","interval_size":0.1,"nb_points":10,"selection":"median"}'
edge_inf = LinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

In [ ]:
config_sup = '{"position":"top","interval_type":"size","interval_nb":10,"nb_points":10,"selection":"median","interval_limits":["percentile(20)","percentile(80)"]}'
edge_sup = LinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"size","interval_nb":10,"nb_points":10,"selection":"median"}'
edge_inf = LinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

Slope correction on LinearEdge

In [ ]:
var, lst = setup(
    var_min=0.0,
    var_max=0.6,
    dry_c0=330.0,
    dry_c1=13.0,
    wet_c0=300,
    wet_c1=-30,
)
plot(var, lst)

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median"}'
edge_sup = LinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median"}'
edge_inf = LinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median","slope_correction":true}'
edge_sup = LinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median","slope_correction":true}'
edge_inf = LinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

Linear edge with variable percentiles

In [ ]:
var, lst = setup(
    var_min=0.0,
    var_max=0.6,
    dry_c0=330.0,
    dry_c1=-13.0,
    wet_c0=300,
    wet_c1=30,
    law="nonuniform",
)
plot(var, lst)

In [ ]:
config_sup = '{"position":"top","interval_type":"size","interval_nb":10,"percentile_bounds":[50,1],"percentile_intervals":[500,100000],"selection":"median"}'
edge_sup = LinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"size","interval_nb":10,"percentile_bounds":[40,2],"percentile_intervals":[500,100000],"selection":"median"}'
edge_inf = LinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

LinearEdge with breakpoint

In [ ]:
# Generate data
var1, lst1 = setup(
    var_min=0.0,
    var_max=0.3,
    dry_c0=325.0,
    dry_c1=37.0,
    dry_c2=0.0,
    wet_c0=310,
    wet_c1=-36,
    wet_c2=0.0,
    dry_cut=0.0,
    wet_cut=0.0,
)
var2, lst2 = setup(
    var_min=0.3,
    var_max=0.6,
    dry_c0=340.0,
    dry_c1=-13.0,
    dry_c2=0.0,
    wet_c0=290,
    wet_c1=30,
    wet_c2=0.0,
    dry_cut=0.0,
    wet_cut=0.0,
)
var = np.concatenate([var1, var2])
lst = np.concatenate([lst1, lst2])
plot(var, lst)

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median","use_breakpoint":true}'
edge_sup = LinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"size","interval_size":0.01,"percentile":2,"selection":"median","use_breakpoint":true}'
edge_inf = LinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median","fit_breakpoint":0.25}'
edge_sup = LinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"size","interval_size":0.01,"percentile":2,"selection":"median","fit_breakpoint":0.3}'
edge_inf = LinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median","use_breakpoint":true,"fit_breakpoint":0.25,"slope_correction":true}'
edge_sup = LinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"size","interval_size":0.05,"percentile":2,"selection":"median","use_breakpoint":true,"fit_breakpoint":0.2,"slope_correction":true}'
edge_inf = LinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

Linear edge with threshold

In [ ]:
var, lst = setup(
    var_min=0.0,
    var_max=0.6,
    dry_c0=330.0,
    dry_c1=-13.0,
    wet_c0=300,
    wet_c1=30,
    dry_cut=0.2,
    wet_cut=0.1,
)
plot(var, lst)

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median"}'
edge_sup = ThresholdLinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"percentile":2,"selection":"min"}'
edge_inf = ThresholdLinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median","use_extremum":false}'
edge_sup = ThresholdLinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"percentile":2,"selection":"min","use_extremum":false}'
edge_inf = ThresholdLinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

Flat linear edge

In [ ]:
var, lst = setup(
    var_min=0.0,
    var_max=0.6,
    dry_c0=330.0,
    dry_c1=-13.0,
    wet_c0=300,
    wet_c1=30,
    # dry_cut=0.2,
    wet_cut=0.15,
)
plot(var, lst)

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median"}'
edge_sup = FlatLinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"percentile":2,"selection":"min"}'
edge_inf = FlatLinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median","use_extremum":false}'
edge_sup = FlatLinearEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"percentile":2,"selection":"min","use_extremum":false}'
edge_inf = FlatLinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

Double linear edge

In [ ]:
# Generate data
var1, lst1 = setup(
    var_min=0.0,
    var_max=0.3,
    dry_c0=325.0,
    dry_c1=37.0,
    dry_c2=0.0,
    wet_c0=310,
    wet_c1=-36,
    wet_c2=0.0,
    dry_cut=0.0,
    wet_cut=0.0,
)
var2, lst2 = setup(
    var_min=0.3,
    var_max=0.6,
    dry_c0=340.0,
    dry_c1=-13.0,
    dry_c2=0.0,
    wet_c0=290,
    wet_c1=30,
    wet_c2=0.0,
    dry_cut=0.0,
    wet_cut=0.0,
)
var = np.concatenate([var1, var2])
lst = np.concatenate([lst1, lst2])
plot(var, lst)

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median"}'
edge_sup = DoubleLinearEdge.model_validate_json(config_sup)
x, y = edge_sup.get_points(var, lst)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"percentile":2,"selection":"min"}'
edge_inf = DoubleLinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median","use_extremum":false}'
edge_sup = DoubleLinearEdge.model_validate_json(config_sup)
x, y = edge_sup.get_points(var, lst)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"percentile":2,"selection":"min","use_extremum":false}'
edge_inf = DoubleLinearEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

Parabolic edge

In [ ]:
var, lst = setup(
    var_min=0.0,
    var_max=0.6,
    dry_c0=330.0,
    dry_c1=-5.0,
    dry_c2=-10.0,
    wet_c0=300,
    wet_c1=30,
    wet_c2=15.0,
    size=100,
)
plot(var, lst)

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":2,"selection":"median"}'
edge_sup = ParabolicEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"percentile":2,"selection":"min"}'
edge_inf = ParabolicEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

Flat regression edge

In [ ]:
var, lst = setup(
    var_min=0.0,
    var_max=0.6,
    dry_c0=330.0,
    dry_c1=-5.0,
    wet_c0=300,
    wet_c1=30,
    size=100,
)
plot(var, lst)

In [ ]:
config_sup = '{"position":"top","interval_type":"density","interval_nb":20,"percentile":10,"selection":"median"}'
edge_sup = FlatRegressionEdge.model_validate_json(config_sup)
edge_sup.fit(var, lst)
print(edge_sup)
config_inf = '{"position":"bottom","interval_type":"density","interval_nb":20,"percentile":10,"selection":"median"}'
edge_inf = FlatRegressionEdge.model_validate_json(config_inf)
edge_inf.fit(var, lst)
print(edge_inf)
plot(var, lst, edges=[edge_sup, edge_inf])

## EF model definition

In [ ]:
def setup_data(
    var_min: float,
    var_max: float,
    dry_c0: float,
    wet_c0: float,
    dry_c1: float,
    wet_c1: float,
    dry_c2: float = 0.0,
    wet_c2: float = 0.0,
    dry_cut: float = 0.0,
    wet_cut: float = 0.0,
    valid: float = 1.0,
    size: int = 200,
    seed: int = 0,
    law: str = "uniform",
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Generate data for tests
    """
    # Seed
    np.random.seed(seed)
    # Lat/lon
    lon = np.linspace(start=1.0, stop=2.0, num=size, dtype=np.float32)
    lat = np.linspace(start=43.0, stop=44.0, num=size, dtype=np.float32)

    # Edge definitions
    def dry_edge(v: float) -> float:
        return dry_c2 * v * v + dry_c1 * v + dry_c0

    def wet_edge(v: float) -> float:
        return wet_c2 * v * v + wet_c1 * v + wet_c0

    if law == "nonuniform":
        df = pd.DataFrame(
            data={
                "var": np.random.triangular(
                    left=var_min, mode=var_min, right=var_max, size=size * size
                )
            }
        )
    else:
        df = pd.DataFrame(
            data={
                "var": np.random.uniform(
                    low=var_min, high=var_max, size=size * size
                )
            }
        )

    df["lst"] = df.apply(
        lambda x: np.random.uniform(wet_edge(x["var"]), dry_edge(x["var"])),
        axis=1,
    )
    dry_lst_cut = dry_edge(dry_cut)
    wet_lst_cut = wet_edge(wet_cut)
    df.loc[(df["var"] <= dry_cut) & (df["lst"] > dry_lst_cut), "lst"] = (
        dry_lst_cut
    )
    df.loc[(df["var"] <= wet_cut) & (df["lst"] < wet_lst_cut), "lst"] = (
        wet_lst_cut
    )
    lst_arr = df["lst"].to_numpy().reshape(size, -1)
    var_arr = df["var"].to_numpy().reshape(size, -1)
    valid_arr = np.random.choice(
        [0, 1], size=(size, size), p=[1 - valid, valid]
    )
    ef_arr = (dry_edge(var_arr) - lst_arr) / (
        dry_edge(var_arr) - wet_edge(var_arr)
    )
    return xr.Dataset(
        data_vars={
            "var": (["lon", "lat"], var_arr),
            "lst": (["lon", "lat"], lst_arr),
            "valid": (["lon", "lat"], valid_arr),
            "ef": (["lon", "lat"], ef_arr),
        },
        coords={
            "lon": ("lon", lon),
            "lat": ("lat", lat),
        },
        attrs={"description": "Test data"},
    )

In [ ]:
def plot_data(
    data: xr.Dataset,
    model: EFModel | None = None,
):
    """
    Plot
    """
    # dimensions
    # gridspec inside gridspec
    _ = plt.figure(constrained_layout=True, figsize=(4, 3))

    ax = plt.gca()
    var = data["var"].data
    lst = data["lst"].data
    # plot all points
    ax.scatter(var, lst, s=10, alpha=1, color="moccasin", clip_on=False)

    # plot model
    if model is not None:
        # edges coordinates
        var_max = np.round(np.ceil(np.nanmax(var) * 10) / 10, decimals=1)
        var_min = np.round(np.floor(np.nanmin(var) * 10) / 10, decimals=1)

        var_arr = np.linspace(var_min, var_max, num=1000)
        tw_arr = model.twet(var_arr)
        td_arr = model.tdry(var_arr)

        ax.plot(
            var_arr,
            td_arr,
            color="tab:red",
            linewidth=3,
            label="dry edge",
            zorder=1,
        )
        ax.plot(
            var_arr,
            tw_arr,
            color="tab:blue",
            linewidth=3,
            label="wet edge",
            zorder=1,
        )

    # Labels
    ax.set_xlabel("Variable")
    ax.set_ylabel("Land Surface Temperature K")

    # title
    ax.set_title("Evaporative fraction method", fontsize=12)

    plt.show()

Uniform data

In [ ]:
data = setup_data(
    var_min=0.0,
    var_max=0.6,
    dry_c0=330.0,
    dry_c1=-13.0,
    wet_c0=300,
    wet_c1=30,
)

In [ ]:
data

In [ ]:
plot_data(data)

Model (LinearEdge, LinearEdge)

In [ ]:
config_model1 = {
    "name": "model1",
    "dry_edge": {
        "type": "LinearEdge",
        "config": {
            "interval_type": "density",
            "interval_nb": 20,
            "percentile": 2,
            "selection": "median",
        },
    },
    "wet_edge": {
        "type": "LinearEdge",
        "config": {
            "interval_type": "size",
            "interval_size": 0.05,
            "percentile": 2,
            "selection": "median",
        },
    },
    "var": "var",
}

In [ ]:
model1 = EFModel.create(config_model1)

In [ ]:
model1.fit(data)

In [ ]:
model1.fit(data, mask="valid")

In [ ]:
plot_data(data, model=model1)

In [ ]:
print(model1)

In [ ]:
model1

In [ ]:
model1.to_dict()

Model (FlatLinearEdge,FlatEdge)

In [ ]:
config_model2 = {
    "name": "model1",
    "dry_edge": {
        "type": "FlatLinearEdge",
        "config": {
            "interval_type": "density",
            "interval_nb": 20,
            "percentile": 2,
            "selection": "median",
        },
    },
    "wet_edge": {
        "type": "FlatEdge",
    },
    "var": "var",
}

In [ ]:
model2 = EFModel.create(config_model2)
model2.fit(data)
print(model2)
plot_data(data, model=model2)

Nonuniform data

In [ ]:
data = setup_data(
    var_min=0.0,
    var_max=0.6,
    dry_c0=330.0,
    dry_c1=-13.0,
    wet_c0=300,
    wet_c1=30,
    size=80,
    law="nonuniform",
)

In [ ]:
data

In [ ]:
plot_data(data)

Model (LinearEdge, LinearEdge)

In [ ]:
config_model1 = {
    "name": "model1",
    "dry_edge": {
        "type": "LinearEdge",
        "config": {
            "interval_type": "size",
            "interval_nb": 10,
            "percentile": 5,
            "selection": "median",
        },
    },
    "wet_edge": {
        "type": "LinearEdge",
        "config": {
            "interval_type": "size",
            "interval_nb": 10,
            "percentile": 5,
            "selection": "median",
        },
    },
    "var": "var",
}

In [ ]:
model1 = EFModel.create(config_model1)
model1.fit(data)
print(model1)
plot_data(data, model=model1)

Model (LinearEdge, LinearEdge) with variable percentiles

In [ ]:
config_model2 = {
    "name": "model1",
    "dry_edge": {
        "type": "LinearEdge",
        "config": {
            "interval_type": "size",
            "interval_nb": 10,
            "percentile_bounds": [5, 1],
            "percentile_intervals": [100, 1000000],
            "selection": "median",
        },
    },
    "wet_edge": {
        "type": "LinearEdge",
        "config": {
            "interval_type": "size",
            "interval_nb": 10,
            "percentile_bounds": [5, 1],
            "percentile_intervals": [100, 1000000],
            "selection": "median",
        },
    },
    "var": "var",
}

In [ ]:
model2 = EFModel.create(config_model2)
model2.fit(data)
print(model2)
plot_data(data, model=model2)